# GP4 ReAct-IR Qwen2.5 QLoRA Cloud Workflow

Colab runtime must persist generated data, reports, checkpoints, adapters, and inference outputs to Google Drive only.

The notebook uses `gp4_finetune_factory_source_bundle.zip` from `$CLOUD_ROOT/bundles/` when present. Otherwise it clones the pushed `codex/gp4-react-ir-cloud-workflow` branch into the ephemeral runtime. Generated data, reports, checkpoints, adapters, inference outputs, and packages stay under `CLOUD_ROOT`.


In [ ]:
import json
import os
import shutil
import subprocess
import zipfile
from datetime import datetime
from pathlib import Path
from google.colab import drive

drive.mount('/content/drive')
DRIVE_ACCOUNT_EMAIL = 'johnwickiller4444@gmail.com'
DRIVE_ROOT = os.environ.get('GP4_DRIVE_ROOT', '/content/drive/MyDrive/gp4_finetune_factory')
SOURCE_BRANCH = os.environ.get('GP4_SOURCE_BRANCH', 'codex/gp4-react-ir-cloud-workflow')
GP4_WS_REPO_URL = os.environ.get('GP4_WS_REPO_URL', 'https://github.com/Hieu-RMX18/gp4_ws.git')
GP4_WS_BRANCH = os.environ.get('GP4_WS_BRANCH', 'ws-deep-rebuild-3526')
RUN_ID = os.environ.get('RUN_ID') or 'gp4-react-v2-300k-' + datetime.utcnow().strftime('%Y%m%d-%H%M')
CLOUD_ROOT = os.environ.get('CLOUD_ROOT') or f'{DRIVE_ROOT}/{RUN_ID}'
PREVIOUS_RUN_ID = os.environ.get('GP4_PREVIOUS_RUN_ID', '').strip()
OLD_DATASET = os.environ.get('GP4_OLD_DATASET', '').strip()
PREVIOUS_ADAPTER = os.environ.get('GP4_PREVIOUS_ADAPTER', '').strip()
if not OLD_DATASET and PREVIOUS_RUN_ID:
    previous_candidate = Path(DRIVE_ROOT) / PREVIOUS_RUN_ID / 'data/validated/accepted_300k.jsonl'
    if previous_candidate.exists():
        OLD_DATASET = str(previous_candidate)
if not PREVIOUS_ADAPTER and PREVIOUS_RUN_ID:
    previous_adapter_candidate = Path(DRIVE_ROOT) / PREVIOUS_RUN_ID / 'models/qwen25_gp4_lora'
    if previous_adapter_candidate.exists():
        PREVIOUS_ADAPTER = str(previous_adapter_candidate)
SOURCE_BUNDLE = f'{CLOUD_ROOT}/bundles/gp4_finetune_factory_source_bundle.zip'
WORK_DIR = Path('/content/gp4_finetune_factory_source')
assert CLOUD_ROOT.startswith('/content/drive/'), 'CLOUD_ROOT must live in Google Drive.'
assert SOURCE_BUNDLE.startswith('/content/drive/'), 'Source bundle must live in Google Drive.'
os.makedirs(f'{CLOUD_ROOT}/reports', exist_ok=True)
os.makedirs(f'{CLOUD_ROOT}/manifests', exist_ok=True)
Path(f'{CLOUD_ROOT}/manifests/drive_account_hint.txt').write_text(DRIVE_ACCOUNT_EMAIL + '\n', encoding='utf-8')
CONFIRMED_DRIVE_ACCOUNT_EMAIL = os.environ.get('GP4_DRIVE_ACCOUNT_CONFIRMED', '').strip()
if not CONFIRMED_DRIVE_ACCOUNT_EMAIL:
    CONFIRMED_DRIVE_ACCOUNT_EMAIL = input('Confirm mounted Google Drive account email: ').strip()
if CONFIRMED_DRIVE_ACCOUNT_EMAIL != DRIVE_ACCOUNT_EMAIL:
    raise RuntimeError(f'Google Drive account confirmation mismatch: expected {DRIVE_ACCOUNT_EMAIL}, got {CONFIRMED_DRIVE_ACCOUNT_EMAIL}')
Path(f'{CLOUD_ROOT}/manifests/drive_account_confirmation.json').write_text(json.dumps({'confirmed': True, 'confirmed_email': CONFIRMED_DRIVE_ACCOUNT_EMAIL, 'expected_email': DRIVE_ACCOUNT_EMAIL, 'method': 'operator_input_after_drive_mount'}, sort_keys=True) + '\n', encoding='utf-8')
os.chdir('/content')
if WORK_DIR.exists():
    shutil.rmtree(WORK_DIR)
if Path(SOURCE_BUNDLE).exists():
    with zipfile.ZipFile(SOURCE_BUNDLE) as archive:
        archive.extractall(WORK_DIR)
else:
    !git clone --branch {SOURCE_BRANCH} --depth 1 https://github.com/Hieu-RMX18/gp4_finetune_factory.git {WORK_DIR}
os.chdir(WORK_DIR)
SEED_PATH = str(WORK_DIR / 'data/seed/gp4_seed_starter.jsonl')
SOURCE_PLAN = str(WORK_DIR / 'docs/superpowers/plans/2026-05-20-gp4-v2-300k-readiness.md')
os.environ['RUN_ID'] = RUN_ID
os.environ['CLOUD_ROOT'] = CLOUD_ROOT
os.environ['SEED_PATH'] = SEED_PATH
os.environ['SOURCE_PLAN'] = SOURCE_PLAN
GP4_WS_EXPECTED_COMMIT = os.environ.get('GP4_WS_EXPECTED_COMMIT', '').strip()
if not GP4_WS_EXPECTED_COMMIT:
    raise RuntimeError('GP4_WS_EXPECTED_COMMIT is required before cloning or reusing the gp4_ws contract snapshot')
if len(GP4_WS_EXPECTED_COMMIT) < 12:
    raise RuntimeError('GP4_WS_EXPECTED_COMMIT must be a full SHA or at least 12 hex characters')
if not OLD_DATASET:
    raise RuntimeError('Set GP4_OLD_DATASET or GP4_PREVIOUS_RUN_ID to reuse a previous accepted_300k.jsonl from Google Drive')
DRIVE_ROOT_PATH = Path(DRIVE_ROOT).resolve(strict=False)
OLD_DATASET_PATH = Path(OLD_DATASET).resolve(strict=False)
if OLD_DATASET_PATH != DRIVE_ROOT_PATH and DRIVE_ROOT_PATH not in OLD_DATASET_PATH.parents:
    raise RuntimeError('GP4_OLD_DATASET must live under the configured Google Drive root')
os.environ['GP4_OLD_DATASET'] = str(OLD_DATASET_PATH)
if not PREVIOUS_ADAPTER:
    raise RuntimeError('Set GP4_PREVIOUS_ADAPTER or GP4_PREVIOUS_RUN_ID to reuse a previous fine-tune adapter from Google Drive')
PREVIOUS_ADAPTER_PATH = Path(PREVIOUS_ADAPTER).resolve(strict=False)
if PREVIOUS_ADAPTER_PATH != DRIVE_ROOT_PATH and DRIVE_ROOT_PATH not in PREVIOUS_ADAPTER_PATH.parents:
    raise RuntimeError('GP4_PREVIOUS_ADAPTER must live under the configured Google Drive root')
os.environ['GP4_PREVIOUS_ADAPTER'] = str(PREVIOUS_ADAPTER_PATH)
GP4_WS_ROOT = (Path(CLOUD_ROOT) / 'contract_snapshots').resolve(strict=False)
GP4_WS = os.environ.get('GP4_WS', '').strip()
if not GP4_WS:
    GP4_WS = f'{CLOUD_ROOT}/contract_snapshots/gp4_ws_{GP4_WS_BRANCH}'
    if not Path(GP4_WS).exists():
        Path(GP4_WS).parent.mkdir(parents=True, exist_ok=True)
        subprocess.run(['git', 'clone', '--branch', GP4_WS_BRANCH, '--depth', '1', GP4_WS_REPO_URL, GP4_WS], check=True)
GP4_WS_PATH = Path(GP4_WS).resolve(strict=False)
if GP4_WS_PATH != GP4_WS_ROOT and GP4_WS_ROOT not in GP4_WS_PATH.parents:
    raise RuntimeError('GP4_WS must live under CLOUD_ROOT/contract_snapshots')
GP4_WS = str(GP4_WS_PATH)
os.environ['GP4_WS'] = GP4_WS
ACTUAL_GP4_WS_COMMIT = subprocess.run(['git', '-C', GP4_WS, 'rev-parse', 'HEAD'], text=True, capture_output=True, check=True).stdout.strip()
if not ACTUAL_GP4_WS_COMMIT.lower().startswith(GP4_WS_EXPECTED_COMMIT.lower()):
    raise RuntimeError(f'GP4_WS commit mismatch: expected {GP4_WS_EXPECTED_COMMIT}, got {ACTUAL_GP4_WS_COMMIT}')
os.environ['GP4_WS_EXPECTED_COMMIT'] = GP4_WS_EXPECTED_COMMIT
print(CLOUD_ROOT)
print('Drive account hint:', DRIVE_ACCOUNT_EMAIL)
print('GP4_WS contract snapshot:', os.environ['GP4_WS'])
print('GP4_WS expected commit:', os.environ['GP4_WS_EXPECTED_COMMIT'])
print('GP4_WS actual commit:', ACTUAL_GP4_WS_COMMIT[:12])
print('Old dataset:', os.environ.get('GP4_OLD_DATASET', '<none>'))
print('Previous adapter:', os.environ.get('GP4_PREVIOUS_ADAPTER', '<none>'))
print(WORK_DIR)


In [ ]:
import os
from google.colab import userdata

for secret_name in ('DEEPSEEK_API_KEY', 'OPENAI_API_KEY', 'OPENAI_BASE_URL', 'OPENAI_MODEL'):
    if not os.environ.get(secret_name):
        try:
            secret_value = userdata.get(secret_name)
        except Exception:
            secret_value = None
        if secret_value:
            os.environ[secret_name] = secret_value
os.environ['DEEPSEEK_BASE_URL'] = os.environ.get('DEEPSEEK_BASE_URL', 'https://api.deepseek.com')
os.environ['OPENAI_MODEL'] = os.environ.get('OPENAI_MODEL', 'gpt-5.4')
os.environ['HF_HOME'] = f'{CLOUD_ROOT}/.cache/huggingface'
os.environ['TRANSFORMERS_CACHE'] = f'{CLOUD_ROOT}/.cache/huggingface/transformers'
os.environ['HF_DATASETS_CACHE'] = f'{CLOUD_ROOT}/.cache/huggingface/datasets'
os.environ['TORCH_HOME'] = f'{CLOUD_ROOT}/.cache/torch'
os.environ['XDG_CACHE_HOME'] = f'{CLOUD_ROOT}/.cache'
os.environ['WANDB_DIR'] = f'{CLOUD_ROOT}/wandb'
os.environ['TMPDIR'] = f'{CLOUD_ROOT}/tmp'
for key in ('HF_HOME', 'TRANSFORMERS_CACHE', 'HF_DATASETS_CACHE', 'TORCH_HOME', 'XDG_CACHE_HOME', 'WANDB_DIR', 'TMPDIR'):
    os.makedirs(os.environ[key], exist_ok=True)


In [ ]:
!python -m pip install -q -r requirements-cloud.txt


In [ ]:
!python scripts/provider_probe.py --provider colab --cloud-root "$CLOUD_ROOT" --report "$CLOUD_ROOT/reports/platform_status_${RUN_ID}.json"


In [ ]:
subprocess.run([
    'python',
    'scripts/colab_readiness_report.py',
    '--cloud-root', CLOUD_ROOT,
    '--run-id', RUN_ID,
    '--gp4-ws', GP4_WS,
    '--old-dataset', os.environ['GP4_OLD_DATASET'],
    '--previous-adapter', os.environ['GP4_PREVIOUS_ADAPTER'],
    '--expected-commit', GP4_WS_EXPECTED_COMMIT,
    '--report', f'{CLOUD_ROOT}/reports/colab_readiness_{RUN_ID}.json',
], check=True)

orchestrator_command = [
    'python',
    'scripts/cloud_orchestrator.py',
    '--run-id', RUN_ID,
    '--cloud-root', CLOUD_ROOT,
    '--seed', SEED_PATH,
    '--source-plan', SOURCE_PLAN,
    '--phases', 'ignored',
    '--preset', 'v2-300k',
]
if os.environ.get('GP4_OLD_DATASET'):
    orchestrator_command.extend(['--old-dataset', os.environ['GP4_OLD_DATASET']])
if os.environ.get('GP4_PREVIOUS_ADAPTER'):
    orchestrator_command.extend(['--previous-adapter', os.environ['GP4_PREVIOUS_ADAPTER']])
subprocess.run(orchestrator_command, check=True)


In [ ]:
!python scripts/audit_cloud_completion.py --cloud-root "$CLOUD_ROOT" --run-id "$RUN_ID" --report "$CLOUD_ROOT/reports/completion_audit_${RUN_ID}.json"


The orchestrator stops automatically unless the previous gate report has `passed=true`. Completion evidence is `$CLOUD_ROOT/reports/acceptance_gate_report_$RUN_ID.json` with `passed=true`, `$CLOUD_ROOT/reports/benchmark_report_${RUN_ID}.html`, `$CLOUD_ROOT/reports/benchmark_report_${RUN_ID}.md` with Maintenance Reference, and package metadata under the same cloud root.
